# NB_bishop_ch07_gradient_descent

**Bishop Chapter 7 — Gradient Descent Optimization**

This notebook implements gradient descent from scratch on the Rosenbrock function, compares batch/SGD/mini-batch, implements momentum and Adam, explores learning rate schedules, and demonstrates batch normalization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_07/NB_bishop_ch07_gradient_descent.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Part 1: Gradient Descent on the Rosenbrock Function

The Rosenbrock function $f(x,y) = (1-x)^2 + 100(y-x^2)^2$ is a classic test for optimization algorithms. Its narrow curved valley makes it challenging for gradient descent.

In [ ]:
# Define Rosenbrock function and its gradient
def rosenbrock(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return dx, dy

# Create contour data
x_grid = np.linspace(-2, 2, 400)
y_grid = np.linspace(-1, 3, 400)
X, Y = np.meshgrid(x_grid, y_grid)
Z = rosenbrock(X, Y)

print(f'Minimum at (1, 1), f(1,1) = {rosenbrock(1, 1)}')

In [ ]:
# Implement vanilla gradient descent from scratch
def gradient_descent(x0, y0, lr=0.001, n_steps=5000):
    path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        dx, dy = rosenbrock_grad(x, y)
        x -= lr * dx
        y -= lr * dy
        path.append((x, y))
    return np.array(path)

# Run with different learning rates
start = (-1.5, 2.0)
paths_lr = {}
for lr in [0.0001, 0.0005, 0.001]:
    path = gradient_descent(*start, lr=lr, n_steps=8000)
    final_f = rosenbrock(path[-1, 0], path[-1, 1])
    paths_lr[f'lr={lr}'] = path
    print(f'lr={lr}: final point ({path[-1,0]:.4f}, {path[-1,1]:.4f}), f={final_f:.6f}')

In [ ]:
# Plot GD trajectories on Rosenbrock contour
fig, ax = plt.subplots(figsize=(9, 6))
ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 30), norm=LogNorm(), cmap='coolwarm', alpha=0.6)
ax.plot(1, 1, 'k*', markersize=15, label='Minimum (1,1)')

colors_gd = ['#1a3a6e', '#cd0000', '#228B22']
for (name, path), c in zip(paths_lr.items(), colors_gd):
    ax.plot(path[::20, 0], path[::20, 1], '-o', color=c, markersize=2, linewidth=1, label=name)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gradient Descent on Rosenbrock Function')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch7_gd_trajectory')
plt.show()

## Part 2: Batch vs SGD vs Mini-batch

We compare three variants on a neural network regression task.

In [ ]:
# Generate regression data
np.random.seed(42)
N = 500
x_data = np.random.uniform(-3, 3, (N, 1)).astype(np.float32)
y_data = (np.sin(x_data) + 0.2 * np.random.randn(N, 1)).astype(np.float32)

print(f'Dataset: {N} samples')

In [ ]:
# Train with different batch sizes
batch_configs = {
    'Full Batch': N,
    'Mini-batch (32)': 32,
    'SGD (1)': 1
}

batch_histories = {}
for name, bs in batch_configs.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mse')
    hist = model.fit(x_data, y_data, epochs=100, batch_size=bs, verbose=0)
    batch_histories[name] = hist.history['loss']
    print(f'{name:>20s}: final loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot batch size comparison
fig, ax = plt.subplots(figsize=(8, 5))
for (name, losses), c in zip(batch_histories.items(), ['#1a3a6e', '#228B22', '#cd0000']):
    ax.plot(losses, label=name, color=c, linewidth=1.5, alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Batch vs Mini-batch vs SGD')
ax.legend()
fig.tight_layout()
plt.show()

## Part 3: Momentum and Adam

Momentum accelerates GD in relevant directions. Adam combines momentum with adaptive learning rates.

In [ ]:
# Implement momentum GD from scratch on Rosenbrock
def gd_momentum(x0, y0, lr=0.001, momentum=0.9, n_steps=5000):
    path = [(x0, y0)]
    x, y = x0, y0
    vx, vy = 0.0, 0.0
    for _ in range(n_steps):
        dx, dy = rosenbrock_grad(x, y)
        vx = momentum * vx - lr * dx
        vy = momentum * vy - lr * dy
        x += vx
        y += vy
        path.append((x, y))
    return np.array(path)

# Implement Adam from scratch on Rosenbrock
def adam_optimizer(x0, y0, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=5000):
    path = [(x0, y0)]
    x, y = x0, y0
    mx, my = 0.0, 0.0
    vx, vy = 0.0, 0.0
    for t in range(1, n_steps + 1):
        dx, dy = rosenbrock_grad(x, y)
        mx = beta1 * mx + (1 - beta1) * dx
        my = beta1 * my + (1 - beta1) * dy
        vx = beta2 * vx + (1 - beta2) * dx**2
        vy = beta2 * vy + (1 - beta2) * dy**2
        mx_hat = mx / (1 - beta1**t)
        my_hat = my / (1 - beta1**t)
        vx_hat = vx / (1 - beta2**t)
        vy_hat = vy / (1 - beta2**t)
        x -= lr * mx_hat / (np.sqrt(vx_hat) + eps)
        y -= lr * my_hat / (np.sqrt(vy_hat) + eps)
        path.append((x, y))
    return np.array(path)

In [ ]:
# Run all three optimizers
start = (-1.5, 2.0)
n_steps = 10000

path_vanilla = gradient_descent(*start, lr=0.0005, n_steps=n_steps)
path_momentum = gd_momentum(*start, lr=0.0003, momentum=0.9, n_steps=n_steps)
path_adam = adam_optimizer(*start, lr=0.005, n_steps=n_steps)

for name, path in [('Vanilla GD', path_vanilla), ('Momentum', path_momentum), ('Adam', path_adam)]:
    f_val = rosenbrock(path[-1, 0], path[-1, 1])
    print(f'{name:>12s}: final ({path[-1,0]:.4f}, {path[-1,1]:.4f}), f = {f_val:.6f}')

In [ ]:
# Plot optimizer comparison on Rosenbrock
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

opt_data = [
    ('Vanilla GD', path_vanilla, '#1a3a6e'),
    ('Momentum', path_momentum, '#cd0000'),
    ('Adam', path_adam, '#228B22')
]

for ax, (name, path, c) in zip(axes, opt_data):
    ax.contour(X, Y, Z, levels=np.logspace(-1, 3.5, 25), norm=LogNorm(), cmap='coolwarm', alpha=0.5)
    step = max(1, len(path) // 200)
    ax.plot(path[::step, 0], path[::step, 1], '-o', color=c, markersize=2, linewidth=0.8)
    ax.plot(1, 1, 'k*', markersize=12)
    ax.set_title(name)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

fig.suptitle('Optimizer Trajectories on Rosenbrock', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch7_optimizer_comparison')
plt.show()

## Part 4: Optimizer Comparison on Neural Network

In [ ]:
# Compare optimizers on actual neural network training
optimizer_configs = {
    'SGD': tf.keras.optimizers.SGD(learning_rate=0.01),
    'SGD+Momentum': tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': tf.keras.optimizers.RMSprop(learning_rate=0.001),
    'Adam': tf.keras.optimizers.Adam(learning_rate=0.001)
}

opt_histories = {}
for name, opt in optimizer_configs.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=opt, loss='mse')
    hist = model.fit(x_data, y_data, epochs=200, batch_size=32, verbose=0)
    opt_histories[name] = hist.history['loss']
    print(f'{name:>15s}: final loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot optimizer comparison on NN
fig, ax = plt.subplots(figsize=(8, 5))
colors_opt = ['#cd0000', '#DAA520', '#8B008B', '#228B22']
for (name, losses), c in zip(opt_histories.items(), colors_opt):
    ax.plot(losses, label=name, color=c, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Optimizer Comparison on NN Regression')
ax.legend()
fig.tight_layout()
plt.show()

## Part 5: Learning Rate Schedules

In [ ]:
# Define learning rate schedules
epochs = 200
steps_per_epoch = N // 32

schedules = {
    'Constant (0.01)': tf.keras.optimizers.schedules.PiecewiseConstantDecay(
        boundaries=[epochs * steps_per_epoch], values=[0.01, 0.01]),
    'Step Decay': tf.keras.optimizers.schedules.PiecewiseConstantDecay(
        boundaries=[50*steps_per_epoch, 100*steps_per_epoch, 150*steps_per_epoch],
        values=[0.01, 0.005, 0.001, 0.0005]),
    'Exponential Decay': tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.01, decay_steps=steps_per_epoch*20, decay_rate=0.5),
    'Cosine Decay': tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.01, decay_steps=epochs*steps_per_epoch)
}

In [ ]:
# Visualize the schedules
steps = np.arange(epochs * steps_per_epoch)
fig, ax = plt.subplots(figsize=(8, 5))
colors_sched = ['#1a3a6e', '#cd0000', '#228B22', '#DAA520']
for (name, sched), c in zip(schedules.items(), colors_sched):
    lr_vals = [sched(s).numpy() for s in steps[::steps_per_epoch]]
    ax.plot(range(0, epochs), lr_vals, label=name, color=c, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedules')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch7_lr_schedule')
plt.show()

In [ ]:
# Train with each schedule
sched_histories = {}
for name, sched in schedules.items():
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=sched), loss='mse')
    hist = model.fit(x_data, y_data, epochs=epochs, batch_size=32, verbose=0)
    sched_histories[name] = hist.history['loss']
    print(f'{name:>25s}: final loss = {hist.history["loss"][-1]:.6f}')

In [ ]:
# Plot training loss for each schedule
fig, ax = plt.subplots(figsize=(8, 5))
for (name, losses), c in zip(sched_histories.items(), colors_sched):
    ax.plot(losses, label=name, color=c, linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Effect of Learning Rate Schedules')
ax.legend()
fig.tight_layout()
plt.show()

## Part 6: Batch Normalization

In [ ]:
# Generate more complex data for BN experiment
from sklearn.datasets import make_moons
X_bn, y_bn = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_bn = X_bn.astype(np.float32)
y_bn = y_bn.astype(np.float32)

print(f'Classification data: {X_bn.shape[0]} samples, {X_bn.shape[1]} features')

In [ ]:
# Compare deep networks with and without batch normalization
def build_deep_model(use_bn=False, n_layers=8, units=64):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(2,)))
    for _ in range(n_layers):
        model.add(tf.keras.layers.Dense(units))
        if use_bn:
            model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation('relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    return model

bn_results = {}
for use_bn, name in [(False, 'Without BN'), (True, 'With BN')]:
    tf.random.set_seed(42)
    model = build_deep_model(use_bn=use_bn)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_bn, y_bn, epochs=100, batch_size=32, validation_split=0.2, verbose=0)
    bn_results[name] = hist.history
    print(f'{name:>15s}: val_acc = {hist.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Plot BN comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for (name, hist), c in zip(bn_results.items(), ['#cd0000', '#228B22']):
    axes[0].plot(hist['loss'], label=f'{name} (train)', color=c, linewidth=1.5)
    axes[0].plot(hist['val_loss'], label=f'{name} (val)', color=c, linewidth=1.5, linestyle='--')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss: Batch Normalization Effect')
axes[0].legend()

for (name, hist), c in zip(bn_results.items(), ['#cd0000', '#228B22']):
    axes[1].plot(hist['accuracy'], label=f'{name} (train)', color=c, linewidth=1.5)
    axes[1].plot(hist['val_accuracy'], label=f'{name} (val)', color=c, linewidth=1.5, linestyle='--')

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy: Batch Normalization Effect')
axes[1].legend()

fig.tight_layout()
plt.show()

## Summary

Key takeaways from Bishop Chapter 7:
- **Gradient descent** converges slowly on ill-conditioned surfaces like Rosenbrock.
- **Mini-batch SGD** offers the best trade-off between noise (regularization) and computational efficiency.
- **Momentum** accelerates convergence along consistent gradient directions.
- **Adam** adapts per-parameter learning rates and is the default optimizer for many tasks.
- **Learning rate schedules** (cosine, exponential) improve final convergence.
- **Batch normalization** stabilizes training of deep networks by normalizing layer inputs.

In [ ]:
print('Notebook complete.')